# Logistic Combination Model

Example notebook for the object recognition task

## Load libraries

In [1]:
import numpy as np
import pandas as pd

from itertools import combinations
from sklearn.linear_model import LogisticRegression

## Define experimental setup

In [ ]:
# Image phase noise (paper, H>M: 80; M>H: 125)
noise_level = 125

# Experiments utilizing confidence (standard), or without confidence (no_confidence)
experiment_type = ['standard', 'no_confidence'][0]

## Prepare human-machine data

In [3]:
# Path to data directory
root_path = '../data/imagenet/'

# List of classifiers to be analyzed
human = pd.read_csv(root_path+'Behavioral Data/human_only_classification_6per_img_export.csv')
machine = pd.read_csv(root_path+'Machine Classifier Predictions/hai_epoch01_model_preds_max_normalized.csv')

human = human[human['noise_level'] == noise_level]
machine = machine[machine['noise_level'] == noise_level]

In [4]:
selected_machines = np.unique(machine['model_name'])
selected_machines

array(['alexnet', 'densenet161', 'googlenet', 'resnet152', 'vgg19'],
      dtype=object)

In [5]:
n_classes = np.unique(human['image_category'])
n_classes

array(['airplane', 'bear', 'bicycle', 'bird', 'boat', 'bottle', 'car',
       'cat', 'chair', 'clock', 'dog', 'elephant', 'keyboard', 'knife',
       'oven', 'truck'], dtype=object)

In [6]:
# Initialize classification 
classification = pd.DataFrame()
classification = human[['image_name','image_category','participant_classification']].copy()
classification.columns = ['image_name','true labels','human']
classification = classification.reset_index(drop=True)

# Initialize confidence
confidence = human[['image_name', 'confidence']].copy()
confidence['confidence'] = -confidence['confidence'].map({'low': 1/3, 'medium': 2/3, 'high': 3/3})
confidence = confidence.reset_index(drop=True)

confidence = pd.concat([confidence, pd.DataFrame([confidence['confidence']]*(len(n_classes)-1)).T], axis=1)
confidence.columns = ['image_name'] + ['human-' + cls for cls in n_classes]

for idx, row in classification.iterrows():
    confidence.loc[idx, 'human-' + row['human']] *= -1


In [7]:
for i in selected_machines:
    
    # Read PPL scores of machine classifiers
    machine_tmp = machine[machine['model_name']==i]
    
    # Get classification results
    machine_classification = machine_tmp[['image_name', 'model_pred']]
    machine_classification.columns = ['image_name', i]
    classification = classification.merge(machine_classification, on='image_name', how='left')
    
    # Define confidence as PPL difference
    machine_confidence = machine_tmp.drop(columns=['noise_type', 'noise_level', 'category', 'model_name', 'model_pred', 'correct'])
    machine_confidence.iloc[:, 1:] = -machine_confidence.iloc[:, 1:]
    machine_confidence = machine_confidence[['image_name'] + list(n_classes)]
    for idx, row in machine_classification.iterrows():
        machine_confidence.loc[idx, row[i]] *= -1
    machine_confidence.columns = ['image_name'] + [i + '-' + cls for cls in n_classes]
    confidence = confidence.merge(machine_confidence, on='image_name', how='left')

In [8]:
# Get all possible combinations of classifiers
all_classifiers = classification.columns[2:].values
all_combinations = []
# for i in range(1, len(all_classifiers)+1):
for i in range(1, 4):
    els = [list(x) for x in combinations(all_classifiers, i)]
    all_combinations.extend(els)
    
all_combinations = all_combinations[:31]
    
# Create a dataframe to store predictions
predictions = pd.DataFrame(columns=all_classifiers)
for i in range(len(all_combinations)):
    for element in all_classifiers:
        predictions.loc[i, element] = element in all_combinations[i]

In [9]:
if experiment_type == 'no_confidence':
    confidence.iloc[:, 1:] = np.sign(confidence.iloc[:, 1:])

## Run logistic combination model

In [10]:
# Leave-one-out cross-validation
for i in range(len(all_combinations)):
    
    tmp_pred = pd.DataFrame()
    
    for j in machine_classification['image_name']:
        
        # Train/test data for current fold
        X_train = confidence[confidence['image_name']!=j]
        y_train = classification[classification['image_name']!=j]['true labels'].values
        
        X_test = confidence[confidence['image_name']==j]
        y_test = classification[classification['image_name']==j]
    
        if i < len(all_classifiers):
            
            # Get prediction accuracy of single classifiers
            acc = y_test[all_classifiers[i]].values == y_test['true labels'].values
            
            # Save prediction accuracy 
            tmp_pred = pd.concat([tmp_pred, pd.DataFrame({'Accuracy': acc})])
            
        else:
            
            selected_columns = [col for col in X_train.columns if col.startswith(tuple(all_combinations[i]))]
            
            # Train logistic regression model
            clf = LogisticRegression(random_state=1, max_iter=100000).fit(X_train[selected_columns], y_train)
            
            # Get prediction accuracy for different teams
            acc = clf.predict(X_test[selected_columns]) == y_test['true labels'].values
            
            # Save prediction accuracy 
            tmp_pred = pd.concat([tmp_pred, pd.DataFrame({'Accuracy': acc})])
    
    predictions.loc[i, 'Accuracy'] = tmp_pred.mean().values
           
# Save predictions 
predictions.to_csv(f'../results/imagenet_{noise_level}_logistic_predictions_{experiment_type}_experiment.csv', index=False)
predictions

,human,alexnet,densenet161,googlenet,resnet152,vgg19,Accuracy
0,True,False,False,False,False,False,0.603398
1,False,True,False,False,False,False,0.549247
2,False,False,True,False,False,False,0.668462
3,False,False,False,True,False,False,0.638348
4,False,False,False,False,True,False,0.628402
5,False,False,False,False,False,True,0.682277
6,True,True,False,False,False,False,0.638072
7,True,False,True,False,False,False,0.695400
8,True,False,False,True,False,False,0.690703
9,True,False,False,False,True,False,0.689736
